# 1.Config/Import

In [1]:
import time
from datetime import datetime
import random
import os
import pickle
import itertools
import gc

# 데이터 분석 라이브러리
import numpy as np
import pandas as pd
import gzip
import zipfile
import math
from tqdm import tqdm
from tqdm import trange
tqdm.pandas()  # <- 이걸 실행해야 progress_apply가 활성화됨

# 케라스 라이브러리
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Embedding, Flatten, Dot, Multiply, Concatenate, Dropout, Dense, Add, Activation,
    MultiHeadAttention, Attention, Lambda, GlobalMaxPooling1D, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam, SGD

# 데이터 분할, 인코더, 성능 평가 라이브러리
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import ParameterGrid

print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')
print(f'tensorflow version: {tf.__version__}')       # 2.13.0 등

pandas version: 1.5.3
numpy version: 1.23.5
tensorflow version: 2.12.0


In [27]:
DATA_SAVE_PATH = 'C:/Users/Bigdata64/Desktop/MK/dataset'
# DATA_SAVE_PATH = '/content/drive/MyDrive/박민경_논문1/dataset/prep'
CATEGORY = 'yelp'
DATASET_NM = 'yelp'

# 변수 설정
dict_col_nm = {
   'amazon': {
        'USER_ID': 'userID',
        'ITEM_ID': 'itemID',
        'RATING_COL': 'rating',
        'REVIEW_COL': 'reviewtext'
      },
   'yelp': {
        'USER_ID': 'user_id',
        'ITEM_ID': 'rest_id',
        'RATING_COL': 'review_stars',
        'REVIEW_COL': 'review_text'
      }
}

# 평균 패딩 시퀀스 길이 할당
dict_mean_seq_len = {
    'book':118,
    'movie':44,
    'yelp':113,
    'office': 51
}

USER_COL = 'user'
ITEM_COL = 'item'
RATING_COL = 'rating'
# REVIEW_COL = dict_col_nm[DATASET_NM]['REVIEW_COL']

mean_seq_length = dict_mean_seq_len[CATEGORY]

print(f"{CATEGORY} average seq length: {mean_seq_length}")

train_path = f'{DATA_SAVE_PATH}/{CATEGORY}_train_dict.pkl'
test_path = f'{DATA_SAVE_PATH}/{CATEGORY}_test_dict.pkl'

yelp average seq length: 113


# 2.load data

In [ ]:
df_input = pd.read_pickle(f"{DATA_SAVE_PATH}/{CATEGORY}_raw_embedding.pkl")

print(df_input.shape)
df_input.head(3)

(472877, 7)


,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[304, 42, 8, 3, 29, 370, 231, 2, 1, 536, 30, 9...","[-1.598509, -0.997596, 0.5327545, -1.525021, 1...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
1,ejFxLGqQcWNLdNByJlIhnQ,XQfwVwDr-v0ZS3_CbbE5Xw,4.0,"[46, 898, 383, 71, 44, 14, 49, 8, 32, 19, 32, ...","[-1.4743725, -1.3923125, -0.2920361, -2.608637...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
2,f7xa0p_1V9lx53iIGN5Sug,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[21, 52, 3, 190, 7, 14, 1, 121, 188, 919, 8, 3...","[-0.21176815, -0.2568603, -1.0278563, -2.40477...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."


## (1)grouping

In [4]:
# count interaction by user
df_user_counts = df_input.groupby("user")["rating"].count().rename("cnt").reset_index()      # ← DataFrame으로 변환됨


# give rank based on the num of  interaction per user
df_user_counts["pct"] = df_user_counts["cnt"].rank(method="first", pct=True)

# assign user group
df_user_counts["user_group"] = pd.cut(
    df_user_counts["pct"],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=[1, 2, 3, 4],
    include_lowest=True
).astype(int)

df_user_counts.head(5)


,user,cnt,pct,user_group
0,--4AjktZiHowEIBCMd4CZA,12,0.684666,3
1,--F7iEaFFPO1UYemj-dUNw,6,0.187627,1
2,--_r6E98SNIrGU7weyNxbw,32,0.920491,4
3,--ccVMj2PN6Z9qtdOdlung,6,0.187658,1
4,--dVKamoZnV2vYwKtWMVVA,14,0.743281,3


In [5]:
# mapping on dictionary
user_to_group = dict(zip(df_user_counts["user"], df_user_counts["user_group"]))

# mapping on df
df_input["user_group"] = df_input["user"].map(user_to_group)

df_input.head(5)

,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart,user_group
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[304, 42, 8, 3, 29, 370, 231, 2, 1, 536, 30, 9...","[-1.598509, -0.997596, 0.5327545, -1.525021, 1...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",4
1,ejFxLGqQcWNLdNByJlIhnQ,XQfwVwDr-v0ZS3_CbbE5Xw,4.0,"[46, 898, 383, 71, 44, 14, 49, 8, 32, 19, 32, ...","[-1.4743725, -1.3923125, -0.2920361, -2.608637...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",4
2,f7xa0p_1V9lx53iIGN5Sug,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[21, 52, 3, 190, 7, 14, 1, 121, 188, 919, 8, 3...","[-0.21176815, -0.2568603, -1.0278563, -2.40477...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",4
3,dCooFVCk8M1nVaQqcfTL3Q,XQfwVwDr-v0ZS3_CbbE5Xw,2.0,"[39, 1, 470, 6, 642, 2, 2531, 2, 4, 429, 33, 4...","[0.82523656, 0.91178995, 0.60730594, -4.156382...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",2
4,gy7Ss1uTpCjbbGsghTvNsw,XQfwVwDr-v0ZS3_CbbE5Xw,2.0,"[9, 3, 208, 119, 16, 22, 26, 19, 382, 17, 64, ...","[-2.3661036, -0.6385349, 0.5271855, -3.6942627...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",3


In [6]:
# split dataframe based on user group
df_g1 = df_input[df_input['user_group']==1]
df_g2 = df_input[df_input['user_group']==2]
df_g3 = df_input[df_input['user_group']==3]
df_g4 = df_input[df_input['user_group']==4]

print(
    f"user group1: {df_g1['user_group'].unique()}", 
    f"user group2: {df_g2['user_group'].unique()}", 
    f"user group3: {df_g3['user_group'].unique()}", 
    f"user group4: {df_g4['user_group'].unique()}", 
    )

user group1: [1] user group2: [2] user group3: [3] user group4: [4]


In [7]:
# save user group data
df_input.to_pickle(f"{DATA_SAVE_PATH}/df_{CATEGORY}_input.pkl")
print("✅ df_input.pkl 저장 완료")

✅ df_input.pkl 저장 완료


In [8]:
with open(f"{DATA_SAVE_PATH}/{CATEGORY}_user_to_group.pkl", "wb") as f:
    pickle.dump(user_to_group, f)

print(f"✅ {CATEGORY}_user_to_group.pkl 저장 완료")


✅ yelp_user_to_group.pkl 저장 완료


In [9]:
with open(f"{DATA_SAVE_PATH}/{CATEGORY}_user_to_group.pkl", "rb") as f:
    user_to_group = pickle.load(f)

## (2)import embedding matrix

In [10]:
# user/item 임베딩 행렬 불러오기
user_embedding_matrix = np.load(f'{DATA_SAVE_PATH}/{CATEGORY}_user_glove_dic.npy', allow_pickle=True)
item_embedding_matrix = np.load(f'{DATA_SAVE_PATH}/{CATEGORY}_item_glove_dic.npy', allow_pickle=True)

user_embedding_matrix.shape, item_embedding_matrix.shape

((50001, 300), (50001, 300))

In [11]:
df_input = pd.read_pickle(f"{DATA_SAVE_PATH}/df_{CATEGORY}_input.pkl")

In [21]:
print(df_input.shape)
df_input.head(5)

(472877, 8)


,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart,user_group
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[304, 42, 8, 3, 29, 370, 231, 2, 1, 536, 30, 9...","[-1.598509, -0.997596, 0.5327545, -1.525021, 1...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",4
1,ejFxLGqQcWNLdNByJlIhnQ,XQfwVwDr-v0ZS3_CbbE5Xw,4.0,"[46, 898, 383, 71, 44, 14, 49, 8, 32, 19, 32, ...","[-1.4743725, -1.3923125, -0.2920361, -2.608637...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",4
2,f7xa0p_1V9lx53iIGN5Sug,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[21, 52, 3, 190, 7, 14, 1, 121, 188, 919, 8, 3...","[-0.21176815, -0.2568603, -1.0278563, -2.40477...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",4
3,dCooFVCk8M1nVaQqcfTL3Q,XQfwVwDr-v0ZS3_CbbE5Xw,2.0,"[39, 1, 470, 6, 642, 2, 2531, 2, 4, 429, 33, 4...","[0.82523656, 0.91178995, 0.60730594, -4.156382...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",2
4,gy7Ss1uTpCjbbGsghTvNsw,XQfwVwDr-v0ZS3_CbbE5Xw,2.0,"[9, 3, 208, 119, 16, 22, 26, 19, 382, 17, 64, ...","[-2.3661036, -0.6385349, 0.5271855, -3.6942627...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321...",3


# 3.Model

## (0)NCF

In [16]:
def Modeling(user_num, item_num, output_dim):

    # Input
    user = Input(shape=(1,), dtype='int64')
    item = Input(shape=(1,), dtype='int64')

    # GMF
    # MF User Vector
    gmf_user = Embedding(user_num, output_dim, input_length=1,name='gmf_user')(user)
    gmf_user = Flatten()(gmf_user)

    # MF Item Vector
    gmf_item = Embedding(item_num, output_dim, input_length=1, name='gmf_item')(item)
    gmf_item = Flatten()(gmf_item)

    # MLP
    # MLP User Vector
    mlp_user = Embedding(user_num, output_dim, input_length=1, name='mlp_user')(user)
    mlp_user = Flatten()(mlp_user)

    # MLP Item Vector
    mlp_item = Embedding(item_num, output_dim, input_length=1,name='mlp_item')(item)
    mlp_item = Flatten()(mlp_item)

    # GMF Layer : Element-wise Product
    gmf_mul = Multiply()([gmf_user, gmf_item])

    # MLP Layer 1
    mlp_concat = Concatenate()([mlp_user, mlp_item])

    # MLP Lyaer 2
    dense1 = Dense(32, activation='relu')(mlp_concat)
    dense2 = Dense(16, activation='relu')(dense1)
    dense3 = Dense(8, activation='relu')(dense2)

    # NeuMF Layer
    neumf_concat = Concatenate()([gmf_mul, dense3])

    # Output Layer
    output_layer = Dense(1, name='output_layer')(neumf_concat)

    model = Model(inputs=[user, item], outputs=output_layer)
    return model

In [17]:
def run_ncf_all_groups(
    df_total, user_to_group, rating_col,
    output_dim=16, batch_size=128, epochs=100, lr=0.0001, seed=42, n_runs=10
):
    results = []

    # 전체 df에서 필요한 컬럼만
    df_total = df_total[['user', 'item', rating_col]].copy()
    df_total['user_group'] = df_total['user'].map(user_to_group)

    for g in [1, 2, 3, 4]:
        print(f"\n==============================")
        print(f"🎯 NCF | Group G{g}")
        print("==============================")

        # ============================
        # ✅ (1) 그룹 데이터 준비 — 1번만
        # ============================
        df_g = df_total[df_total['user_group'] == g].reset_index(drop=True)
        print(f"📊 interactions: {len(df_g)}")

        # Label Encoding (group당 1번)
        user_encoder = LabelEncoder()
        item_encoder = LabelEncoder()

        df_g['user_enc'] = user_encoder.fit_transform(df_g['user'])
        df_g['item_enc'] = item_encoder.fit_transform(df_g['item'])

        user_num = df_g['user_enc'].nunique()
        item_num = df_g['item_enc'].nunique()

        # train / test split (group당 1번)
        train_data, test_data = train_test_split(
            df_g,
            test_size=0.2,
            random_state=seed,
            shuffle=True
        )

        train_users = train_data[['user_enc']].values
        train_items = train_data[['item_enc']].values
        train_y = train_data[rating_col].values

        test_users = test_data[['user_enc']].values
        test_items = test_data[['item_enc']].values
        test_y = test_data[rating_col].values

        # ============================
        # ✅ (2) 모델 학습만 n번 반복
        # ============================
        for run in range(n_runs):
            print(f"\n🔁 Group G{g} | Run {run+1}/{n_runs}")

            # # run별 seed (초기화만 다르게)
            # seed_everything(seed + run)

            model = Modeling(
                user_num=user_num,
                item_num=item_num,
                output_dim=output_dim
            )

            sgd = SGD(learning_rate=lr)
            model.compile(
                optimizer=sgd,
                loss='MSE',
                metrics=['mse', 'mae']
            )

            early_stopping = EarlyStopping(
                monitor='val_loss',
                min_delta=0.001,
                patience=5,
                verbose=1,
                mode='min',
                restore_best_weights=True
            )

            # 학습
            model.fit(
                x=[train_users, train_items],
                y=train_y,
                validation_split=0.125,
                batch_size=batch_size,
                epochs=epochs,
                callbacks=[early_stopping],
                verbose=1
            )

            # 평가
            predictions = model.predict([test_users, test_items]).flatten()

            MAE = mean_absolute_error(test_y, predictions)
            MSE = mean_squared_error(test_y, predictions)
            RMSE = np.sqrt(MSE)
            MAPE = mean_absolute_percentage_error(test_y, predictions) * 100

            metrics = {
                'group': g,
                'run': run + 1,
                'MAE': MAE,
                'MSE': MSE,
                'RMSE': RMSE,
                'MAPE': MAPE,
                'num_users': user_num,
                'num_items': item_num,
                'num_interactions': len(df_g)
            }

            print("📊 Metrics:", metrics)
            results.append(metrics)

            # 그래프/메모리 정리
            tf.keras.backend.clear_session()

    return pd.DataFrame(results)


In [18]:
def save_results_to_csv(df_results, prefix="results"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    filename = f"{prefix}_{timestamp}.csv"
    df_results.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"📁 Saved: {filename}")
    
    return filename

In [28]:
df_ncf_results = run_ncf_all_groups(
    df_total=df_input,          # 원본 df_total
    user_to_group=user_to_group,
    rating_col=RATING_COL,
    output_dim=16,
    batch_size=128,
    epochs=100,
    lr=0.0001, 
    n_runs=10
)

print(df_ncf_results)



🎯 NCF | Group G1
📊 interactions: 43209

🔁 Group G1 | Run 1/10
Epoch 1/100
237/237 [==============================] - 1s 2ms/step - loss: 15.8153 - mse: 15.8153 - mae: 3.7454 - val_loss: 15.0984 - val_mse: 15.0984 - val_mae: 3.6540
Epoch 2/100
237/237 [==============================] - 0s 1ms/step - loss: 13.9243 - mse: 13.9243 - mae: 3.4834 - val_loss: 13.1736 - val_mse: 13.1736 - val_mae: 3.3805
Epoch 3/100
237/237 [==============================] - 0s 1ms/step - loss: 11.9771 - mse: 11.9771 - mae: 3.1917 - val_loss: 11.1475 - val_mse: 11.1475 - val_mae: 3.0662
Epoch 4/100
237/237 [==============================] - 0s 1ms/step - loss: 9.9040 - mse: 9.9040 - mae: 2.8599 - val_loss: 8.9721 - val_mse: 8.9721 - val_mae: 2.7300
Epoch 5/100
237/237 [==============================] - 0s 1ms/step - loss: 7.6970 - mse: 7.6970 - mae: 2.5159 - val_loss: 6.6921 - val_mse: 6.6921 - val_mae: 2.3523
Epoch 6/100
237/237 [==============================] - 0s 1ms/step - loss: 5.4980 - mse: 5.4980 - ma

In [29]:
save_results_to_csv(df_ncf_results, prefix=f"g1_g4_NCF_{CATEGORY}")

📁 Saved: g1_g4_NCF_yelp_20251218_1637.csv


'g1_g4_NCF_yelp_20251218_1637.csv'

## (1)Proposed Model

In [25]:
def point_wise_feed_forward_network(d_model, dff):
  return tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='linear'),
      tf.keras.layers.Dense(d_model, activation='linear')
  ])

class CoAttentionBlock(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dff, dropout_rate=0.1, epsilon=1e-6, input_dim=None, **kwargs):
        super(CoAttentionBlock, self).__init__(**kwargs)
        self.multi_head_attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, value_dim=key_dim)
        self.dropout = Dropout(dropout_rate)
        self.add = Add()
        self.layer_norm = tf.keras.layers.LayerNormalization(epsilon = epsilon)
        self.ffn = point_wise_feed_forward_network(input_dim, dff)
        self.flatten = Flatten()


    def call(self, inputs):
        text_expanded, image_expanded = inputs
        co_attention = self.multi_head_attention(text_expanded, image_expanded, image_expanded)
        co_attention = self.dropout(co_attention)
        co_attention = self.add([co_attention, text_expanded])
        co_attention = self.layer_norm(co_attention)

        co_attention_ffn = self.ffn(co_attention)
        co_attention_ffn = self.dropout(co_attention_ffn)
        co_attention_ffn = self.add([co_attention_ffn, co_attention])
        co_attention_ffn = self.layer_norm(co_attention_ffn)

        # return self.flatten(co_attention_ffn)
        return co_attention_ffn

class SelfAttentionBlock(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dff, dropout_rate=0.1,  epsilon=1e-6, input_dim=None, **kwargs):
        super(SelfAttentionBlock, self).__init__(**kwargs)
        self.multi_head_attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, value_dim=key_dim)
        self.dropout = Dropout(dropout_rate)
        self.add = Add()
        self.layer_norm = tf.keras.layers.LayerNormalization(epsilon = epsilon)
        self.ffn = point_wise_feed_forward_network(input_dim, dff)
        self.flatten = Flatten()


    def call(self, inputs):
        information_expanded = inputs
        self_attention = self.multi_head_attention(information_expanded, information_expanded, information_expanded)
        self_attention = self.dropout(self_attention)
        self_attention = self.add([self_attention, information_expanded])
        self_attention = self.layer_norm(self_attention)

        self_attention_ffn = self.ffn(self_attention)
        self_attention_ffn = self.dropout(self_attention_ffn)
        self_attention_ffn = self.add([self_attention_ffn, self_attention])
        self_attention_ffn = self.layer_norm(self_attention_ffn)

        # return self.flatten(self_attention_ffn)
        return self_attention_ffn

In [26]:
def proposed_model(input_dim, textrank_bool = False, bart_bool = False,
                   textrank_user_embedding_matrix = None,
                   textrank_item_embedding_matrix = None,
                   output_dim = 300, num_heads = 4, dff = 1024,
                   fusion_version = 'gmu', gmu_activation = 'sigmoid',
                   mlp_depth=2, mlp_hidden_dim=128, dropout_rate = 0.2):
    """
    [input]
      - textrank_bool: textrank 요약을 반영하는지 여부
      - bart_bool: bart 요약을 반영하는지 여부
      - input_dim: 임베딩 층의 input_dim, 요약의 max_seq_length를 의미함          (※ 각 데이터셋의 평균 패딩 시퀀스 길이임)
      - output_dim: 각 textrank/bart 요약 임베딩 차원                          (=추출/추상형 요약 간 결합 전에 맞춰주어야 하는 차원의 크기)
      - fusion_version: 추출/추상형 요약 간의 결합 방식                           (ex. gmu, concat, inner-product)
      - gmu_activation: gmu로 결합할 때 활성화함수
      - mlp_hidden_dim: mlp 은닉층 차원
      - mlp_depth: mlp 은닉층 깊이
      - dropout_rate: dropout 비율

    [output]
      - model: 모델 객체
    """
    # 0. attn block
    key_dim = output_dim // num_heads

    self_attn_block = SelfAttentionBlock(
        num_heads=num_heads, key_dim=key_dim,
        dropout_rate=dropout_rate,
        dff=dff,
        input_dim=output_dim              # 여기서 input_dim으로 output_dim을 넘겨줘야 함
    )
    
    co_attn_block = CoAttentionBlock(
        num_heads=num_heads, key_dim=key_dim,
        dropout_rate=dropout_rate,
        dff=dff, input_dim=output_dim
    )

    if textrank_bool == True:
        user_textrank_input = Input(shape=(input_dim,), dtype='int32', name='user_textrank_input')
        item_textrank_input = Input(shape=(input_dim,), dtype='int32', name='item_textrank_input')

        # 1. user/item embedding & concat
        user_textrank_embedding = Embedding(
            input_dim = len(textrank_user_embedding_matrix),
            output_dim = 300, input_length = input_dim,
            weights = [textrank_user_embedding_matrix])(user_textrank_input)
        item_textrank_embedding = Embedding(
            input_dim = len(textrank_item_embedding_matrix),
            output_dim = 300, input_length = input_dim,
            weights = [textrank_item_embedding_matrix])(item_textrank_input)

        user_textrank_embedding = GlobalMaxPooling1D()(user_textrank_embedding)
        item_textrank_embedding = GlobalMaxPooling1D()(item_textrank_embedding)

        textrank_concat = Concatenate()([user_textrank_embedding, item_textrank_embedding])
        textrank_proj = Dense(output_dim, activation='linear')(textrank_concat)

        # 2. self multi-head attn
        textrank_input = self_attn_block(tf.expand_dims(textrank_proj, axis=1))      # (batch, 1, output_dim)
        textrank_input = tf.squeeze(textrank_input, axis=1)
    else:
        textrank_input = None

    if bart_bool == True:
        # 1. user/item embedding & concat
        user_bart_input = Input(shape=(768,), name='user_bart_input')
        item_bart_input = Input(shape=(768,), name='item_bart_input')

        bart_concat = Concatenate()([user_bart_input, item_bart_input])
        bart_proj = Dense(output_dim, activation='linear')(bart_concat)

        # 2. self multi-head attn
        bart_proj_expanded = tf.expand_dims(bart_proj, axis=1)                            # (batch, 1, output_dim)
        bart_input = self_attn_block(bart_proj_expanded)
        bart_input = tf.squeeze(bart_input, axis=1)
    else:
        bart_input = None


    # 3. Fusion
    if textrank_bool and bart_bool:
        ## 1) gmu: concat > gate > gated sum
        if fusion_version.lower() == 'gmu':
            concat_inputs = Concatenate(name='concat_inputs')([textrank_input, bart_input])
            gate_sum = Dense(output_dim, activation=gmu_activation, name='gmu_gate')(concat_inputs)

            tanh_textrank = Activation('tanh')(textrank_input)
            tanh_bart = Activation('tanh')(bart_input)

            gmu_output = Add()([
                  Multiply()([gate_sum, tanh_textrank]),
                  Multiply()([Lambda(lambda x: 1 - x)(gate_sum), tanh_bart])
              ])
            combined = gmu_output

        ## 2) concat
        elif fusion_version.lower() == 'concat':
            combined = Concatenate()([textrank_input, bart_input])

        ## 3) inner_product
        elif fusion_version.lower().startswith('inner'):
            combined = Dot(axes=-1)([textrank_input, bart_input])

        ## 4) element-wise
        elif fusion_version.lower().startswith('element'):
            combined = Multiply()([textrank_input, bart_input])
        
        ## 5) attn
        elif fusion_version.lower().startswith('at'):
            # co-attn 위해 다시 expand
            textrank_input_exp = tf.expand_dims(textrank_input, axis=1)  # shape: (None, 1, 300)
            bart_input_exp = tf.expand_dims(bart_input, axis=1)

            co_attn_tr = co_attn_block((textrank_input_exp, bart_input_exp))
            co_attn_br = co_attn_block((bart_input_exp, textrank_input_exp))
            combined = Concatenate(name = 'co_attn_output')([co_attn_tr, co_attn_br])
            combined = tf.squeeze(combined, axis=1)
            
        else:
            raise ValueError("Invalid version")

    elif textrank_bool:
        combined = textrank_input
    elif bart_bool:
        combined = bart_input
    else:
        raise ValueError("At least one of textrank_bool or bart_bool must be True.")


    # 4. MLP -> 평점 예측
    ## (combined의 차원이 'cag' 퓨전 시 2*output_dim이 될 수 있으므로, MLP가 이를 처리할 수 있도록 설계되었는지 확인해야 합니다.)
    mlp_hidden_units = [mlp_hidden_dim // (2 ** i) for i in range(mlp_depth)]
    x = combined
    for i, units in enumerate(mlp_hidden_units):
        x = Dense(units, activation='linear', name=f'mlp_dense_{i+1}')(x)
        if dropout_rate > 0:
            x = Dropout(dropout_rate, name=f'dropout_{i+1}')(x)
    rating_pred = Dense(1, activation='relu', name='rating_output')(x)


    # 5. 모델 구성
    input_list = []
    if textrank_bool == True:
        input_list += [user_textrank_input, item_textrank_input]
    if bart_bool == True:
        input_list += [user_bart_input, item_bart_input]

    model = Model(inputs=input_list, outputs=rating_pred, name=f'{fusion_version}_rating_model')

    return model

## (2)parameter

In [12]:
# 기본 고정값
default_params = {
    'learning_rate': 1e-4,
    'batch_size': 128,
    'dropout_rate': 0.2,
    'mlp_hidden_dim': 128,
    'mlp_depth': 2,
    'embedding_dim': 300,
    'input_dim': mean_seq_length,
    'fusion_version': 'gmu'
}

## (3)training

In [27]:
def run_experiment_once(
    user_embedding_matrix, item_embedding_matrix,
    train_inputs, train_y,
    test_inputs, test_y,
    default_params,
    fusion_version='gmu'
):
    """한 번의 모델 학습 + 평가를 수행하고 test_metrics를 반환"""

    model = proposed_model(
        textrank_bool=True, bart_bool=True,
        textrank_user_embedding_matrix=user_embedding_matrix,
        textrank_item_embedding_matrix=item_embedding_matrix,
        fusion_version=fusion_version,
        input_dim=default_params['input_dim'],
        output_dim=default_params['embedding_dim'],
        dropout_rate=default_params['dropout_rate'],
        mlp_hidden_dim=default_params['mlp_hidden_dim'],
        mlp_depth=default_params['mlp_depth']
    )

    model.compile(
        optimizer=Adam(learning_rate=default_params['learning_rate']),
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[tf.keras.metrics.MeanAbsoluteError(), tf.keras.metrics.MeanSquaredError()]
    )

    early_stopping = EarlyStopping(
        monitor='val_loss', mode='min', patience=5,
        restore_best_weights=True, verbose=0
    )

    history = model.fit(
        train_inputs,
        train_y,
        validation_split=0.125,
        epochs=100,
        batch_size=default_params['batch_size'],
        callbacks=[early_stopping],
        verbose=1
    )

    predicted = model.predict(test_inputs, verbose=0)

    test_metrics = {
        'MAE': mean_absolute_error(test_y, predicted),
        'MSE': mean_squared_error(test_y, predicted),
        'RMSE': np.sqrt(mean_squared_error(test_y, predicted)),
        'MAPE': 100 * mean_absolute_percentage_error(test_y, predicted)
    }

    return test_metrics


In [28]:
def run_experiment_n_times(
    n,
    user_embedding_matrix, item_embedding_matrix,
    train_inputs, train_y,
    test_inputs, test_y,
    default_params,
    fusion_version='gmu'
):
    results = []

    for i in range(n):
        print(f"🔄 Running experiment {i+1}/{n} ...")

        metrics = run_experiment_once(
            user_embedding_matrix, item_embedding_matrix,
            train_inputs, train_y,
            test_inputs, test_y,
            default_params,
            fusion_version=fusion_version
        )

        metrics['run'] = i + 1  # 몇 번째 실행인지 추가
        results.append(metrics)

    df_results = pd.DataFrame(results)
    
    return df_results

In [29]:
def run_group_experiment(
    group_num,
    df_input,
    user_to_group,
    user_embedding_matrix, item_embedding_matrix,
    default_params,
    n_runs=1   # 반복 횟수 지정 가능
):
    print(f"\n============================")
    print(f"🔍 Running Group G{group_num}")
    print("============================\n")

    # STEP 1: df_input에 group 태그 달기
    df_input['user_group'] = df_input['user'].map(user_to_group)

    # STEP 2: 선택한 그룹만 추출
    df_g = df_input[df_input['user_group'] == group_num].reset_index(drop=True)

    # STEP 3: train/test split
    df_train, df_test = train_test_split(
        df_g, test_size=0.2, random_state=42
    )

    # STEP 4: embedding 준비
    train_inputs = [
        np.vstack(df_train['user_embedding_textrank']),
        np.vstack(df_train['item_embedding_textrank']),
        np.vstack(df_train['user_embedding_bart']),
        np.vstack(df_train['item_embedding_bart']),
    ]
    train_y = df_train['rating'].values

    test_inputs = [
        np.vstack(df_test['user_embedding_textrank']),
        np.vstack(df_test['item_embedding_textrank']),
        np.vstack(df_test['user_embedding_bart']),
        np.vstack(df_test['item_embedding_bart']),
    ]
    test_y = df_test['rating'].values

    # STEP 5: 지정된 횟수(n_runs) 만큼 반복 실행
    group_results = []
    for run in range(n_runs):
        print(f"\n➡️  G{group_num} Run {run+1}/{n_runs}")
        metrics = run_experiment_once(
            user_embedding_matrix, item_embedding_matrix,
            train_inputs, train_y,
            test_inputs, test_y,
            default_params,
            fusion_version='gmu'
        )
        metrics['group'] = group_num
        metrics['run'] = run+1
        group_results.append(metrics)

    return group_results


In [30]:
def run_single_group_n_times(
    group_num,
    df_input,
    user_to_group,
    user_embedding_matrix,
    item_embedding_matrix,
    default_params,
    n_runs=5,
    test_size=0.2,
    random_state=42
):
    """
    특정 user group(G1~G4 중 하나)에 대해 n번 학습/평가 수행
    """
    print(f"\n==============================")
    print(f"🎯 Running Group G{group_num} for {n_runs} runs")
    print("==============================\n")

    # 1️⃣ group 태깅
    df = df_input.copy()
    df["user_group"] = df["user"].map(user_to_group)

    # 2️⃣ 특정 그룹만 필터링 (★ split 이전)
    df_g = df[df["user_group"] == group_num].reset_index(drop=True)

    print(f"📊 Group G{group_num} interactions: {len(df_g)}")

    # 3️⃣ train / test split
    df_train, df_test = train_test_split(
        df_g,
        test_size=test_size,
        random_state=random_state
    )

    # 4️⃣ X / y 생성 (컬럼명은 네 데이터에 맞게 수정!)
    train_inputs = [
        np.vstack(df_train["user_embedding_textrank"]),
        np.vstack(df_train["item_embedding_textrank"]),
        np.vstack(df_train["user_embedding_bart"]),
        np.vstack(df_train["item_embedding_bart"]),
    ]
    train_y = df_train["rating"].values

    test_inputs = [
        np.vstack(df_test["user_embedding_textrank"]),
        np.vstack(df_test["item_embedding_textrank"]),
        np.vstack(df_test["user_embedding_bart"]),
        np.vstack(df_test["item_embedding_bart"]),
    ]
    test_y = df_test["rating"].values

    # sanity check
    assert len(test_y) == test_inputs[0].shape[0], "❌ X/y length mismatch!"

    # 5️⃣ n번 반복 학습
    results = []

    for run in range(n_runs):
        print(f"\n🔁 G{group_num} | Run {run+1}/{n_runs}")

        metrics = run_experiment_once(
            user_embedding_matrix,
            item_embedding_matrix,
            train_inputs,
            train_y,
            test_inputs,
            test_y,
            default_params,
            fusion_version="gmu"
        )

        metrics["group"] = group_num
        metrics["run"] = run + 1
        results.append(metrics)

    return pd.DataFrame(results)


In [31]:
def run_all_groups(
    df_input,
    user_to_group,
    user_embedding_matrix, item_embedding_matrix,
    default_params,
    n_runs=1
):
    all_results = []

    for g in [1, 2, 3, 4]:
        group_res = run_group_experiment(
            g, df_input, user_to_group,
            user_embedding_matrix, item_embedding_matrix,
            default_params,
            n_runs=n_runs
        )
        all_results.extend(group_res)

    df_results = pd.DataFrame(all_results)
    return df_results


In [19]:
df_results = run_all_groups(
    df_input=df_input,
    user_to_group=user_to_group,   # 기존에 만든 dict
    user_embedding_matrix=user_embedding_matrix,
    item_embedding_matrix=item_embedding_matrix,
    default_params=default_params,
    n_runs=10   # 각 그룹을 몇 번 반복할지
)

print(df_results)



🔍 Running Group G1


➡️  G1 Run 1/10
Epoch 1/100
237/237 [==============================] - 56s 227ms/step - loss: 2.2425 - mean_absolute_error: 1.2024 - mean_squared_error: 2.2425 - val_loss: 1.6465 - val_mean_absolute_error: 1.0919 - val_mean_squared_error: 1.6465
Epoch 2/100
237/237 [==============================] - 52s 221ms/step - loss: 1.8081 - mean_absolute_error: 1.0934 - mean_squared_error: 1.8081 - val_loss: 1.7891 - val_mean_absolute_error: 0.9733 - val_mean_squared_error: 1.7891
Epoch 3/100
237/237 [==============================] - 52s 221ms/step - loss: 1.7052 - mean_absolute_error: 1.0566 - mean_squared_error: 1.7052 - val_loss: 1.6773 - val_mean_absolute_error: 0.9519 - val_mean_squared_error: 1.6773
Epoch 4/100
237/237 [==============================] - 52s 221ms/step - loss: 1.6901 - mean_absolute_error: 1.0509 - mean_squared_error: 1.6901 - val_loss: 1.4856 - val_mean_absolute_error: 0.9577 - val_mean_squared_error: 1.4856
Epoch 5/100
237/237 [=====================

In [32]:
# 특정 그룹만 n번 돌리기
df_results = run_single_group_n_times(
    group_num=3,
    df_input=df_input,
    user_to_group=user_to_group,
    user_embedding_matrix=user_embedding_matrix,
    item_embedding_matrix=item_embedding_matrix,
    default_params=default_params,
    n_runs=5
)

print(df_results)


🎯 Running Group G3 for 5 runs

📊 Group G3 interactions: 85139

🔁 G3 | Run 1/5
Epoch 1/100
466/466 [==============================] - 106s 223ms/step - loss: 1.9590 - mean_absolute_error: 1.1302 - mean_squared_error: 1.9590 - val_loss: 1.5794 - val_mean_absolute_error: 0.9699 - val_mean_squared_error: 1.5794
Epoch 2/100
466/466 [==============================] - 104s 222ms/step - loss: 1.7148 - mean_absolute_error: 1.0604 - mean_squared_error: 1.7148 - val_loss: 1.5254 - val_mean_absolute_error: 0.9589 - val_mean_squared_error: 1.5254
Epoch 3/100
466/466 [==============================] - 104s 223ms/step - loss: 1.6390 - mean_absolute_error: 1.0355 - mean_squared_error: 1.6390 - val_loss: 1.4663 - val_mean_absolute_error: 0.9518 - val_mean_squared_error: 1.4663
Epoch 4/100
466/466 [==============================] - 104s 223ms/step - loss: 1.5993 - mean_absolute_error: 1.0200 - mean_squared_error: 1.5993 - val_loss: 1.4514 - val_mean_absolute_error: 0.9442 - val_mean_squared_error: 1.45

In [33]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

df_results.to_csv(f"results_{CATEGORY}_{timestamp}.csv", index=False, encoding="utf-8-sig")

In [ ]:
save_results_to_csv(df_results, prefix=f"g1_g4_{CATEGORY}")

📁 Saved: g1_g4_yelp_20251217_1540.csv


'g1_g4_yelp_20251217_1540.csv'